# pinn_dispersion_from_mat.ipynb — PINN Dispersion from Saved Results

**Approach 2 of 2 — Finite Lattice (PINN-based), Stage 2 only**

> **No PINN retraining required.**  
> This notebook loads pre-computed results from `PINN_ndofs_nmasses/pinn_ndof_chain_sim.ipynb`
> saved as a `.mat` file, then performs the spectral analysis.

## Workflow

```
Stage 1  (pinn_ndof_chain_sim.ipynb)
   Run PINN segments → stitch x_n(t) → save_pinn_results('pinn_results.mat', ...)

Stage 2  (this notebook)
   load_pinn_results('pinn_results.mat')
   → resample to uniform dt
   → 2-D FFT  →  S(k, ω)
   → heatmap  +  DOS  +  per-band ridges
```

## Why this is the "real" dispersion

| | Bloch (Approach 1) | PINN finite lattice (this notebook) |
|---|---|---|
| Lattice size | Infinite | Finite (N unit cells as simulated) |
| Boundary conditions | Periodic (exact) | Actual BCs of the simulation |
| Impact timing | Determined by single-cell orbit | Exact root-finding per segment |
| Spatial non-uniformity | Assumed identical cells | Each cell evolves freely |
| Multi-band | Fundamental branch only | Full harmonic structure visible |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'PINN_ndofs_nmasses'))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    clip_spectrum,
    compute_dos,
    detect_band_gaps,
    extract_ridge_in_band,
)
from dispersion_curve import linear_dispersion, plot_dispersion_heatmap

mpl.rcParams.update({
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'pdf.fonttype': 42,
})
FS = 20
LW = 2.0
print('Modules loaded.')

## Load .mat file

Set `mat_path` to the file saved by `pinn_ndof_chain_sim.ipynb`.

In [ ]:
mat_path = '../PINN_ndofs_nmasses/pinn_results.mat'   # ← adjust if needed

t_total, x_PINN_total, params = load_pinn_results(mat_path)

# ── Extract parameters ─────────────────────────────────────────────────────
n_dof = int(params.get('n_dof', x_PINN_total.shape[1]))
mx    = float(params.get('mx', 1.0))
my    = float(params.get('my', 0.1))
K     = float(params.get('k',  100.0))
D     = float(params.get('D',  1.0))
r     = float(params.get('r',  0.7))

omega_max_lin = 2.0 * np.sqrt(K / mx)

print(f'\nParameters from .mat:')
print(f'  n_dof={n_dof}, mx={mx}, my={my}, K={K}, D={D}, r={r}')
print(f'  ω_max_lin = {omega_max_lin:.4f} rad/s')

## Resample to uniform time grid

PINN segments have variable lengths → the stitched `t_total` is piecewise-uniform.
The 2-D FFT requires a globally uniform grid.

In [ ]:
# x_PINN_total is (N_time, n_dof); transpose to (n_dof, N_time) for FFT
X_raw = x_PINN_total.T

n_harmonic    = 4      # resolve up to 4th harmonic band
pts_per_cycle = 10     # samples per shortest cycle

t_uniform, X_uniform, dt, omega_nyquist = resample_to_uniform(
    t_total, X_raw, omega_max_lin,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
)

print(f'Uniform grid : {X_uniform.shape}  (n_dof × N_time)')
print(f'dt           : {dt:.5f} s')
print(f'ω_nyquist    : {omega_nyquist:.3f} rad/s')
print(f'ω resolution : {2*np.pi/(t_uniform[-1]-t_uniform[0]):.4f} rad/s')

## 2-D FFT → dispersion spectrum S(k, ω)

In [ ]:
k_pos, omega_pos, spectrum = compute_spectrum(t_uniform, X_uniform, skip_transient=0.15)

print(f'k range  : {k_pos[0]:.3f} → {k_pos[-1]:.3f} rad/unit-cell')
print(f'ω range  : {omega_pos[0]:.3f} → {omega_pos[-1]:.3f} rad/s')
print(f'Spectrum : {spectrum.shape}')

## Full-band dispersion heatmap

- White dashed — linear acoustic branch  
- Cyan dotted  — effective impact resonance $\omega_\mathrm{eff}$ and harmonics

In [ ]:
# Estimate effective impact frequency from time-span / number of segments
# (conservative: use omega_max_lin as fallback if no impact data saved)
omega_eff     = omega_max_lin / 3.0   # rough initial estimate; refine below
n_bands_shown = 4
omega_max_plot = n_bands_shown * max(omega_eff, omega_max_lin)

k_c, omega_c, spec_c = clip_spectrum(k_pos, omega_pos, spectrum, omega_max_plot)

S_dB = 10.0 * np.log10(spec_c + 1e-30)
S_dB -= S_dB.max()
S_dB  = np.clip(S_dB, -40.0, 0.0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(k_c / np.pi, omega_c, S_dB.T,
                   cmap='inferno', shading='auto', vmin=-40.0, vmax=0.0)
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Spectral power (dB)', fontsize=FS - 4)
cbar.ax.tick_params(labelsize=FS - 6)

# Linear baseline overlay
k_line = np.linspace(0, np.pi, 300)
ax.plot(k_line / np.pi,
        linear_dispersion(k_line, K, mx),
        'w--', lw=LW, label=r'Linear  ($D\!\to\!\infty$)')

ax.set_xlabel(r'Wavenumber  $k/\pi$',         fontsize=FS, labelpad=8)
ax.set_ylabel(r'Frequency  $\omega$  (rad/s)', fontsize=FS, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, omega_max_plot)
ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
ax.tick_params(axis='both', labelsize=FS - 2)
ax.legend(fontsize=FS - 6, loc='upper left', framealpha=0.6)
ax.set_title(f'PINN finite-lattice dispersion  (N={n_dof}, D={D}, r={r})',
             fontsize=FS - 4)
plt.tight_layout(pad=1.5)
plt.savefig('pinn_mat_dispersion_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## Density of States (DOS) — automatic band-gap detection

In [ ]:
dos, dos_dB = compute_dos(spec_c)
gap_freqs, band_freqs = detect_band_gaps(omega_c, dos_dB,
                                         dip_threshold_dB=5.0, min_distance=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(omega_c, dos_dB, 'k-', lw=1.5, label='DOS')
for gf in gap_freqs:
    ax.axvline(gf, color='red', lw=1.2, ls='--', alpha=0.85,
               label=f'gap {gf:.2f} rad/s')
ax.axvline(omega_max_lin, color='gray', lw=1.5, ls='-.',
           label=r'$\omega_{lin,max}$')
ax.set_xlabel(r'$\omega$ (rad/s)', fontsize=FS)
ax.set_ylabel('DOS (dB, norm.)',   fontsize=FS)
ax.set_xlim(0, omega_max_plot)
ax.set_ylim(-42, 2)
ax.tick_params(labelsize=FS - 3)
ax.legend(fontsize=FS - 7)
ax.set_title('Density of States', fontsize=FS - 3)

ax2 = axes[1]
ax2.plot(dos_dB, omega_c, 'k-', lw=1.5)
ax2.fill_betweenx(omega_c, dos_dB, -42, alpha=0.25, color='steelblue')
for gf in gap_freqs:
    ax2.axhline(gf, color='red', lw=1.2, ls='--', label=f'{gf:.2f} rad/s')
ax2.set_ylabel(r'$\omega$ (rad/s)', fontsize=FS)
ax2.set_xlabel('DOS (dB)',          fontsize=FS)
ax2.set_ylim(0, omega_max_plot)
ax2.set_xlim(-42, 2)
ax2.tick_params(labelsize=FS - 3)
if len(gap_freqs):
    ax2.legend(fontsize=FS - 7)
ax2.set_title('DOS (horizontal)', fontsize=FS - 3)

plt.tight_layout()
plt.savefig('pinn_mat_dos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pass-band centres (rad/s): {np.round(band_freqs, 3)}')
print(f'Band-gap centres  (rad/s): {np.round(gap_freqs,  3)}')

## Multi-band ridge extraction

Per-band dominant ω(k) curves, separated by the auto-detected gap frequencies.

In [ ]:
band_edges = np.concatenate([[0.5], np.sort(gap_freqs), [omega_max_plot]])
palette    = plt.cm.plasma(np.linspace(0.1, 0.9, max(len(band_edges) - 1, 1)))

fig, ax = plt.subplots(figsize=(8, 6))

# Linear baseline
k_line = np.linspace(0, np.pi, 300)
ax.plot(k_line / np.pi,
        linear_dispersion(k_line, K, mx),
        'k--', lw=LW, label=r'Linear  ($D\!\to\!\infty$)')

for b, (lo, hi) in enumerate(zip(band_edges[:-1], band_edges[1:])):
    i_lo = np.searchsorted(omega_c, lo)
    i_hi = np.searchsorted(omega_c, hi)
    if i_hi <= i_lo:
        continue
    omega_ridge = extract_ridge_in_band(
        k_c, omega_c[i_lo:i_hi], spec_c[:, i_lo:i_hi], omega_min=lo)
    valid = ~np.isnan(omega_ridge)
    if valid.any():
        ax.plot(k_c[valid] / np.pi, omega_ridge[valid],
                'o-', color=palette[b], lw=LW, ms=6,
                label=f'Band {b+1}  ({lo:.1f}–{hi:.1f} rad/s)')

for gf in gap_freqs:
    ax.axhspan(gf * 0.97, gf * 1.03, color='red', alpha=0.12)

ax.axhline(omega_max_lin, color='gray', lw=1.2, ls='-.',
           label=r'$\omega_{lin,max}$')

ax.set_xlabel(r'Wavenumber  $k/\pi$',         fontsize=FS, labelpad=8)
ax.set_ylabel(r'Frequency  $\omega$  (rad/s)', fontsize=FS, labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, omega_max_plot)
ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
ax.yaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
ax.tick_params(axis='both', labelsize=FS - 2)
ax.legend(fontsize=FS - 6, loc='upper left', framealpha=0.85)
ax.set_title(f'Multi-band PINN dispersion  (N={n_dof}, D={D}, r={r})',
             fontsize=FS - 4)
plt.tight_layout(pad=1.5)
plt.savefig('pinn_mat_multiband.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n{'Band':>6}  {'ω window (rad/s)':>22}  {'k pts with ridge':>18}")
for b, (lo, hi) in enumerate(zip(band_edges[:-1], band_edges[1:])):
    i_lo = np.searchsorted(omega_c, lo)
    i_hi = np.searchsorted(omega_c, hi)
    ridge = extract_ridge_in_band(k_c, omega_c[i_lo:i_hi], spec_c[:, i_lo:i_hi])
    nv    = int((~np.isnan(ridge)).sum())
    print(f"{b+1:>6}  {lo:>10.2f} – {hi:<10.2f}  {nv:>18}")